# KNN Model Training — Influencer-Campaign Match Prediction

**Features:** influencer_category, followers_count, engagement_rate, campaign_category, audience_match_score, previous_performance

**Labels:** Strong Match, Moderate Match, Low Match

**Artifacts saved:** `knn_model.pkl`, `scaler.pkl`, `encoders.pkl`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

## 1. Load Dataset

In [ ]:
df = pd.read_csv('influencer_campaign_dataset.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())
print('\nLabel distribution:')
print(df['match_label'].value_counts())

## 2. Handle Missing Values

In [ ]:
numeric_cols = ['followers_count', 'engagement_rate', 'audience_match_score', 'previous_performance']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    median = df[col].median()
    df[col].fillna(median, inplace=True)
    print(f'Filled {col} NaN with median: {median}')

cat_cols = ['influencer_category', 'campaign_category']
for col in cat_cols:
    mode = df[col].mode()[0]
    df[col].fillna(mode, inplace=True)
    print(f'Filled {col} NaN with mode: {mode}')

df.dropna(subset=['match_label'], inplace=True)
print(f'\nMissing after cleaning: {df.isnull().sum().sum()}')

## 3. Label Encoding

In [ ]:
le_inf = LabelEncoder()
le_camp = LabelEncoder()
le_label = LabelEncoder()

df['inf_enc'] = le_inf.fit_transform(df['influencer_category'])
df['camp_enc'] = le_camp.fit_transform(df['campaign_category'])
df['label_enc'] = le_label.fit_transform(df['match_label'])

print('Influencer categories:', list(le_inf.classes_))
print('Campaign categories:', list(le_camp.classes_))
print('Labels:', list(le_label.classes_))

## 4. Feature Scaling & Train/Test Split

In [ ]:
features = ['inf_enc', 'followers_count', 'engagement_rate', 'camp_enc', 'audience_match_score', 'previous_performance']
X = df[features].values
y = df['label_enc'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 5. Train KNN Model

In [ ]:
knn = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='euclidean')
knn.fit(X_train, y_train)
print('Model trained!')

## 6. Evaluate Model

In [ ]:
y_pred = knn.predict(X_test)

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred, average="weighted"):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}')
print(f'\n{classification_report(y_test, y_pred, target_names=le_label.classes_)}')

## 7. Save All Artifacts Separately

In [ ]:
# Save KNN model
joblib.dump(knn, 'knn_model.pkl')
print('Saved: knn_model.pkl')

# Save fitted StandardScaler
joblib.dump(scaler, 'scaler.pkl')
print('Saved: scaler.pkl')

# Save all fitted LabelEncoders as a dictionary
encoders = {
    'influencer_category': le_inf,
    'campaign_category': le_camp,
    'match_label': le_label,
}
joblib.dump(encoders, 'encoders.pkl')
print('Saved: encoders.pkl')

print('\nAll artifacts saved successfully!')

## 8. Load Artifacts & Verify

In [ ]:
# Load each artifact back from disk
loaded_model = joblib.load('knn_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
loaded_encoders = joblib.load('encoders.pkl')

print('Loaded model type:', type(loaded_model).__name__)
print('Loaded scaler type:', type(loaded_scaler).__name__)
print('Loaded encoder keys:', list(loaded_encoders.keys()))
print('\nScaler means:', np.round(loaded_scaler.mean_, 4))
print('Influencer classes:', list(loaded_encoders['influencer_category'].classes_))
print('Campaign classes:', list(loaded_encoders['campaign_category'].classes_))
print('Label classes:', list(loaded_encoders['match_label'].classes_))

## 9. Full Prediction Workflow Using Loaded Artifacts

In [ ]:
def predict_match(influencer_category, followers_count, engagement_rate,
                  campaign_category, audience_match_score, previous_performance):
    """Complete prediction using the 3 saved artifacts."""
    model = joblib.load('knn_model.pkl')
    scaler = joblib.load('scaler.pkl')
    encoders = joblib.load('encoders.pkl')

    le_inf = encoders['influencer_category']
    le_camp = encoders['campaign_category']
    le_label = encoders['match_label']

    # 1. Encode categorical inputs
    try:
        inf_enc = le_inf.transform([influencer_category])[0]
    except ValueError:
        inf_enc = 0
    try:
        camp_enc = le_camp.transform([campaign_category])[0]
    except ValueError:
        camp_enc = 0

    # 2. Scale features
    features = np.array([[inf_enc, followers_count, engagement_rate,
                          camp_enc, audience_match_score, previous_performance]])
    features_scaled = scaler.transform(features)

    # 3. Predict
    prediction_enc = model.predict(features_scaled)[0]
    prediction = le_label.inverse_transform([prediction_enc])[0]

    # 4. Confidence via predict_proba
    probabilities = model.predict_proba(features_scaled)[0]
    confidence = float(np.max(probabilities))

    return {
        'prediction': prediction,
        'confidence': f'{confidence * 100:.0f}%',
        'probabilities': {
            le_label.classes_[i]: f'{p * 100:.1f}%'
            for i, p in enumerate(probabilities)
        },
    }

# Test predictions
test_cases = [
    ('Fitness', 50000, 6.5, 'Fitness', 85, 90),
    ('Technology', 5000, 1.2, 'Fashion', 15, 20),
    ('Food', 120000, 4.5, 'Food', 55, 60),
]

for inf_cat, fol, eng, camp_cat, aud, prev in test_cases:
    result = predict_match(inf_cat, fol, eng, camp_cat, aud, prev)
    print(f'{inf_cat} vs {camp_cat}: {result["prediction"]} ({result["confidence"]})')